# Notebook 03c — Modelos Lineales (Verdes + Azules)

**Target**: `log1p(Tumor_Telomere fusion rate)`  
**Features verdes**: firstResponse, treatment_agrupado, Mesenchimal_origin  
**Features azules añadidas**: cancer_type, biopsySite, hasSystemicPreTreatment, hasRadiotherapyPreTreatment, treatmentGiven, treatment1  
**Modelos**: Lasso → Ridge → Elastic Net

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

import os

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print(f'CWD: {os.getcwd()}')
print('Librerías cargadas OK')

## 1. Carga y unión de datos

In [ ]:
df = pd.read_csv('datos_limpios_modelo_a.csv')

# Features verdes
df_verde = pd.read_excel('revision_color_code.xlsx', sheet_name='Sheet1',
                         usecols=['Patient_ID_IF_mod', 'firstResponse', 'treatment'])
df = df.merge(df_verde, on='Patient_ID_IF_mod', how='left')

# Features azules
df_azul = pd.read_excel('revision_color_code.xlsx', sheet_name='Sheet1',
                        usecols=['Patient_ID_IF_mod', 'biopsySite', 'hasSystemicPreTreatment',
                                 'hasRadiotherapyPreTreatment', 'treatmentGiven', 'treatment1'])
df = df.merge(df_azul, on='Patient_ID_IF_mod', how='left')

# Mesenquimal
df_mes = pd.read_excel('Tumor_mesenchimal.xlsx')
df_mes.columns = ['cancer_type_code', 'Mesenchimal_origin']
df = df.merge(df_mes, on='cancer_type_code', how='left')

print(f'Shape final: {df.shape}')

## 2. Target

In [ ]:
TARGET = 'Tumor_Telomere fusion rate (Gt-end/coverage*purity)'
y = np.log1p(df[TARGET])
print(f'Skewness target log1p: {y.skew():.2f}')

## 3. Preparación de features

In [ ]:
# Verdes: NaN → Sin_dato
df['firstResponse'] = df['firstResponse'].fillna('Sin_dato')
df['Mesenchimal_origin'] = df['Mesenchimal_origin'].fillna('Sin_dato')
top15 = df['treatment'].value_counts().head(15).index.tolist()
df['treatment_agrupado'] = df['treatment'].apply(
    lambda x: x if x in top15 else ('Sin_dato' if pd.isna(x) else 'Otro')
)

# Azules: NaN → Sin_dato
df['hasSystemicPreTreatment'] = df['hasSystemicPreTreatment'].fillna('Sin_dato')
df['hasRadiotherapyPreTreatment'] = df['hasRadiotherapyPreTreatment'].fillna('Sin_dato')
df['treatmentGiven'] = df['treatmentGiven'].fillna('Sin_dato')

# biopsySite: 291 valores únicos → top 10 + Otro
top10_biopsy = df['biopsySite'].value_counts().head(10).index.tolist()
df['biopsySite_agrupado'] = df['biopsySite'].apply(
    lambda x: x if x in top10_biopsy else ('Sin_dato' if pd.isna(x) else 'Otro')
)

# treatment1: 122 valores únicos → top 10 + Otro
top10_t1 = df['treatment1'].value_counts().head(10).index.tolist()
df['treatment1_agrupado'] = df['treatment1'].apply(
    lambda x: x if x in top10_t1 else ('Sin_dato' if pd.isna(x) else 'Otro')
)

# cancer_type: ya está en datos_limpios (51 valores únicos)
print('Features preparadas OK')
print(f'cancer_type valores únicos: {df["cancer_type"].nunique()}')

In [ ]:
# Dummies de todas las features (verdes + azules)
cols_features = ['firstResponse', 'treatment_agrupado', 'Mesenchimal_origin',
                 'cancer_type', 'biopsySite_agrupado', 'hasSystemicPreTreatment',
                 'hasRadiotherapyPreTreatment', 'treatmentGiven', 'treatment1_agrupado']

X = pd.get_dummies(df[cols_features], drop_first=True)
print(f'Shape X: {X.shape} | Shape y: {y.shape}')
print(f'Total features: {X.shape[1]}')

## 4. Split + escalado

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print(f'Train: {X_train_sc.shape} | Test: {X_test_sc.shape}')

## 5. Función de evaluación

In [ ]:
def evaluar_modelo(nombre, modelo, X_tr, X_te, y_tr, y_te, feature_names):
    modelo.fit(X_tr, y_tr)
    y_pred_log  = modelo.predict(X_te)
    # Deshacer log1p → métricas en espacio original
    y_pred_orig = np.expm1(y_pred_log)
    y_te_orig   = np.expm1(np.array(y_te))

    r2   = r2_score(y_te_orig, y_pred_orig)
    mae  = mean_absolute_error(y_te_orig, y_pred_orig)
    rmse = np.sqrt(mean_squared_error(y_te_orig, y_pred_orig))
    cv   = cross_val_score(modelo, X_tr, y_tr, cv=5, scoring='r2').mean()

    sep = '=' * 50
    print(sep)
    print(f'  {nombre}')
    print(sep)
    print(f'  R²  (test, espacio original):  {r2:.4f}')
    print(f'  R²  (CV-5, espacio log):       {cv:.4f}')
    print(f'  MAE  (espacio original):        {mae:.4f}')
    print(f'  RMSE (espacio original):        {rmse:.4f}')

    coefs = pd.Series(modelo.coef_, index=feature_names)
    nz = coefs[coefs != 0].sort_values(key=abs, ascending=False)
    print(f'  Coefs no nulos: {len(nz)} / {len(coefs)}')
    if len(nz) > 0:
        print(nz.head(10).to_string())

    return {'nombre': nombre, 'R2_test': r2, 'R2_cv': cv,
            'MAE': mae, 'RMSE': rmse, 'y_pred': y_pred_orig}

## 6. Lasso

In [ ]:
res_lasso = evaluar_modelo('Lasso (alpha=0.001)',
    Lasso(alpha=0.001, max_iter=10000, random_state=42),
    X_train_sc, X_test_sc, y_train, y_test, X.columns)

## 6b. Tratamientos — coeficientes positivos y negativos (Lasso, target: TUMOR)

In [ ]:
# Re-entrenamos Lasso con los mismos parámetros para tener acceso a los coeficientes
modelo_lasso_coefs = Lasso(alpha=0.001, max_iter=10000, random_state=42)
modelo_lasso_coefs.fit(X_train_sc, y_train)
coefs = pd.Series(modelo_lasso_coefs.coef_, index=X.columns)

# Variables de tratamiento (excluimos cancer_type que es tipo de tumor)
prefijos_trat = ('firstResponse_', 'treatment_agrupado_', 'Mesenchimal_origin_',
                 'biopsySite_agrupado_', 'hasSystemicPreTreatment_',
                 'hasRadiotherapyPreTreatment_', 'treatmentGiven_', 'treatment1_agrupado_')
coefs_trat = coefs[[c for c in coefs.index if c.startswith(prefijos_trat)]]
coefs_trat_nz = coefs_trat[coefs_trat != 0].sort_values(ascending=False)

n_total = sum(c.startswith(prefijos_trat) for c in X.columns)
print(f'=== TRATAMIENTOS — Lasso (target: fusiones en TUMOR) ===')
print(f'Variables de tratamiento con coef. no nulo: {len(coefs_trat_nz)} de {n_total}')
print()
print('POSITIVOS (tratamiento asociado a MÁS fusiones en tumor):')
pos = coefs_trat_nz[coefs_trat_nz > 0]
for var, coef in pos.items():
    print(f'  {var:55s}  coef={coef:+.4f}')
print()
print('NEGATIVOS (tratamiento asociado a MENOS fusiones en tumor):')
neg = coefs_trat_nz[coefs_trat_nz < 0]
for var, coef in neg.items():
    print(f'  {var:55s}  coef={coef:+.4f}')

In [ ]:
coefs_lasso = pd.Series(modelo_lasso_coefs.coef_, index=X.columns)
ct = coefs_lasso[[c for c in coefs_lasso.index if c.startswith('cancer_type_')]]
ct_nz = ct[ct != 0].sort_values(ascending=False)
print(ct_nz.to_string())

## 7. Ridge

In [ ]:
res_ridge = evaluar_modelo('Ridge (alpha=1.0)',
    Ridge(alpha=1.0, random_state=42),
    X_train_sc, X_test_sc, y_train, y_test, X.columns)

## 8. Elastic Net

In [ ]:
res_enet = evaluar_modelo('Elastic Net (alpha=0.001, l1_ratio=0.5)',
    ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10000, random_state=42),
    X_train_sc, X_test_sc, y_train, y_test, X.columns)

## 9. Comparativa

In [ ]:
resultados = pd.DataFrame([res_lasso, res_ridge, res_enet]).set_index('nombre')
print('=== COMPARATIVA — LINEALES VERDES + AZULES ===')
print(resultados.drop('y_pred', axis=1).round(4).to_string())
print()
print(f'Mejor modelo: {resultados["R2_test"].idxmax()}')
print()
print('--- Comparativa con solo verdes (03a) ---')
print('Lasso:       R2_test=0.0576')
print('Ridge:       R2_test=0.0551')
print('Elastic Net: R2_test=0.0571')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, res in zip(axes, [res_lasso, res_ridge, res_enet]):
    y_pred = np.array(res['y_pred'])
    y_real = np.expm1(np.array(y_test))   # deshacer log1p
    ax.scatter(y_real, y_pred, alpha=0.3, s=10, color='steelblue')
    lim = max(y_real.max(), y_pred.max())
    ax.plot([0, lim], [0, lim], 'r--', linewidth=1)
    ax.set_xlabel('Real (espacio original)')
    ax.set_ylabel('Predicho (espacio original)')
    ax.set_title(f"{res['nombre']}\nR²={res['R2_test']:.4f}")

plt.tight_layout()
plt.savefig('notebooks/img/img_lineales_azules.png', dpi=150, bbox_inches='tight')
plt.close()
print('Guardado: notebooks/img/img_lineales_azules.png')

## 10. Exploración de alpha — ¿importa la fuerza del castigo?

Hasta ahora Lasso usaba `alpha=0.001` y Ridge usaba `alpha=1.0` — el valor por defecto de scikit-learn, nunca elegido conscientemente. Probamos 8 valores distintos para cada modelo y vemos cuál da el mejor R².

In [ ]:
alphas = [0.0001, 0.001, 0.01, 0.1, 0.5, 1.0, 10.0, 100.0]

resultados_ridge = []
resultados_lasso = []

for a in alphas:
    # Ridge
    m_ridge = Ridge(alpha=a, random_state=42)
    m_ridge.fit(X_train_sc, y_train)
    r2_r = r2_score(y_test, m_ridge.predict(X_test_sc))
    cv_r = cross_val_score(m_ridge, X_train_sc, y_train, cv=5, scoring='r2').mean()
    resultados_ridge.append({'alpha': a, 'R2_test': r2_r, 'R2_cv': cv_r})

    # Lasso
    m_lasso = Lasso(alpha=a, max_iter=10000, random_state=42)
    m_lasso.fit(X_train_sc, y_train)
    r2_l = r2_score(y_test, m_lasso.predict(X_test_sc))
    cv_l = cross_val_score(m_lasso, X_train_sc, y_train, cv=5, scoring='r2').mean()
    nz   = np.sum(m_lasso.coef_ != 0)
    resultados_lasso.append({'alpha': a, 'R2_test': r2_l, 'R2_cv': cv_l, 'coefs_no_nulos': nz})

df_ridge = pd.DataFrame(resultados_ridge)
df_lasso = pd.DataFrame(resultados_lasso)

print('=== RIDGE — R² según alpha ===')
print(df_ridge.round(4).to_string(index=False))
mejor_ridge = df_ridge.loc[df_ridge['R2_test'].idxmax()]
print(f'\nMejor Ridge: alpha={mejor_ridge["alpha"]}  R2_test={mejor_ridge["R2_test"]:.4f}')

print()
print('=== LASSO — R² según alpha ===')
print(df_lasso.round(4).to_string(index=False))
mejor_lasso = df_lasso.loc[df_lasso['R2_test'].idxmax()]
print(f'\nMejor Lasso: alpha={mejor_lasso["alpha"]}  R2_test={mejor_lasso["R2_test"]:.4f}  coefs={mejor_lasso["coefs_no_nulos"]:.0f}/103')

## 11. Elastic Net — exploración de l1_ratio

El `l1_ratio` controla la mezcla entre Lasso (L1) y Ridge (L2). Aquí lo variamos manteniendo `alpha=0.001` fijo para ver si alguna mezcla mejora el resultado.

In [ ]:
l1_ratios = [0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0]

resultados_enet = []
for ratio in l1_ratios:
    m = ElasticNet(alpha=0.001, l1_ratio=ratio, max_iter=10000, random_state=42)
    m.fit(X_train_sc, y_train)
    r2 = r2_score(y_test, m.predict(X_test_sc))
    cv = cross_val_score(m, X_train_sc, y_train, cv=5, scoring='r2').mean()
    nz = np.sum(m.coef_ != 0)
    resultados_enet.append({'l1_ratio': ratio, 'R2_test': r2, 'R2_cv': cv, 'coefs_no_nulos': nz})

df_enet = pd.DataFrame(resultados_enet)

print('=== ELASTIC NET — exploración de l1_ratio (alpha=0.001 fijo) ===')
print('  l1_ratio=0.1 → casi Ridge puro | l1_ratio=1.0 → Lasso puro')
print()
print(df_enet.round(4).to_string(index=False))
print()
mejor_enet = df_enet.loc[df_enet['R2_test'].idxmax()]
print(f'Mejor Elastic Net: l1_ratio={mejor_enet["l1_ratio"]}  R2_test={mejor_enet["R2_test"]:.4f}')

## 12. Gráfico — evolución del R² según alpha

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Ridge: R² según alpha
axes[0].semilogx(df_ridge['alpha'], df_ridge['R2_test'], 'o-', color='steelblue', label='R² test')
axes[0].semilogx(df_ridge['alpha'], df_ridge['R2_cv'],   's--', color='orange',   label='R² CV-5')
axes[0].axhline(y=res_lasso['R2_test'], color='green', linestyle=':', linewidth=1.5,
                label=f'Lasso ref = {res_lasso["R2_test"]:.4f}')
axes[0].set_xlabel('Alpha (escala log)')
axes[0].set_ylabel('R²')
axes[0].set_title('Ridge — R² según alpha')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Lasso: R² según alpha
axes[1].semilogx(df_lasso['alpha'], df_lasso['R2_test'], 'o-', color='steelblue', label='R² test')
axes[1].semilogx(df_lasso['alpha'], df_lasso['R2_cv'],   's--', color='orange',   label='R² CV-5')
axes[1].set_xlabel('Alpha (escala log)')
axes[1].set_ylabel('R²')
axes[1].set_title('Lasso — R² según alpha')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Elastic Net: R² según l1_ratio
axes[2].plot(df_enet['l1_ratio'], df_enet['R2_test'], 'o-', color='steelblue', label='R² test')
axes[2].plot(df_enet['l1_ratio'], df_enet['R2_cv'],   's--', color='orange',   label='R² CV-5')
axes[2].set_xlabel('l1_ratio  (0 = Ridge puro → 1 = Lasso puro)')
axes[2].set_ylabel('R²')
axes[2].set_title('Elastic Net — R² según l1_ratio')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('notebooks/img/img_exploracion_alpha.png', dpi=150, bbox_inches='tight')
plt.close()
print('Guardado: notebooks/img/img_exploracion_alpha.png')

## 13. Resumen final — mejor alpha por modelo

In [ ]:
print('=== RESUMEN FINAL — mejor alpha por modelo ===')
print()
print(f'  Lasso       alpha={mejor_lasso["alpha"]}   R2_test={mejor_lasso["R2_test"]:.4f}   coefs={mejor_lasso["coefs_no_nulos"]:.0f}/103')
print(f'  Ridge       alpha={mejor_ridge["alpha"]}   R2_test={mejor_ridge["R2_test"]:.4f}   coefs=103/103')
print(f'  Elastic Net l1_ratio={mejor_enet["l1_ratio"]}  R2_test={mejor_enet["R2_test"]:.4f}   coefs={mejor_enet["coefs_no_nulos"]:.0f}/103')
print()
todos = {
    'Lasso':       mejor_lasso['R2_test'],
    'Ridge':       mejor_ridge['R2_test'],
    'Elastic Net': mejor_enet['R2_test'],
}
ganador = max(todos, key=todos.get)
print(f'Ganador: {ganador}  (R2={todos[ganador]:.4f})')
print()
print('¿Cambia el ganador respecto a los valores por defecto?')
print(f'  Antes → Lasso 0.0846 | Ridge 0.0727 | Elastic Net 0.0806')
print(f'  Ahora → Lasso {mejor_lasso["R2_test"]:.4f} | Ridge {mejor_ridge["R2_test"]:.4f} | Elastic Net {mejor_enet["R2_test"]:.4f}')

## 14. Validacion por Cross-Validation (5-fold) — sobre todos los datos

Hasta ahora el R2=0.085 viene de un solo grupo de test fijo (random_state=42).
Para confirmar que es un resultado solido y no depende de ese grupo concreto,
aplicamos 5-fold CV sobre el dataset completo (4535 pacientes):
cada vuelta usa un 20% distinto como test y el 80% restante para entrenar.
El escalado se hace DENTRO de cada vuelta para no contaminar el test.

Referencia actual (un solo split 80/20, seed=42):
  Lasso R2=0.0846 | Ridge R2=0.0727 | Elastic Net R2=0.0806

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score

kf = KFold(n_splits=5, shuffle=True, random_state=42)
sep = "-"

modelos_cv = {
    "Lasso       (alpha=0.001)":         Lasso(alpha=0.001, max_iter=10000, random_state=42),
    "Ridge       (alpha=100.0)":         Ridge(alpha=100.0, random_state=42),
    "Elastic Net (alpha=0.001, l1=0.5)": ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10000, random_state=42),
}

print("=== CROSS-VALIDATION 5-fold -- dataset completo (n=4535) ===")
print(f"  {'Modelo':<40}  {'Media R2':>9}  {'Std':>7}  {'Min':>7}  {'Max':>7}")
print(f"  {sep*40}  {sep*9}  {sep*7}  {sep*7}  {sep*7}")

for nombre, modelo in modelos_cv.items():
    pipe = Pipeline([("scaler", StandardScaler()), ("model", modelo)])
    scores = cross_val_score(pipe, X, y, cv=kf, scoring="r2")
    print(f"  {nombre:<40}  {scores.mean():>9.4f}  {scores.std():>7.4f}  {scores.min():>7.4f}  {scores.max():>7.4f}")

print()
print("Referencia (un solo split seed=42): Lasso=0.0846 | Ridge=0.0727 | ElasticNet=0.0806")
